# Real-World AI Projects

**Case study:** Retail operations intelligence capstone. Synthetic data only. Run cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data") if Path("data/customers_synthetic.csv").exists() else Path(".")
customers = pd.read_csv(DATA / "customers_synthetic.csv")
orders = pd.read_csv(DATA / "orders_synthetic.csv", parse_dates=["order_date"])
print("Customers:", customers.shape, "Orders:", orders.shape)
display(customers.head())

## Data quality and exploratory checks

In [ ]:
assert customers.customer_id.is_unique
assert orders.order_id.is_unique
assert customers.churned.isin([0,1]).all()
print(customers.isna().sum().to_string())
print("Churn rate:", round(customers.churned.mean(),3))
customers.groupby("plan").churned.agg(["mean","count"]).plot(kind="bar",subplots=True,figsize=(9,3),title=["Churn rate","Customers"]); plt.tight_layout(); plt.show()

## Predictive baseline and evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, recall_score
cols=["age","tenure_months","monthly_spend","support_tickets","engagement_score"]
X=customers[cols];y=customers.churned
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)).fit(Xtr,ytr)
score=clf.predict_proba(Xte)[:,1]
print("Test ROC AUC:",round(roc_auc_score(yte,score),3))
print("Recall @ 0.5:",round(recall_score(yte,score>=.5,zero_division=0),3))
print("Dummy accuracy:",round(DummyClassifier(strategy="most_frequent").fit(Xtr,ytr).score(Xte,yte),3))

## Build KPI table and interactive Plotly dashboard

In [ ]:
import plotly.express as px
completed=orders.loc[orders.status.eq("completed")].copy()
completed["revenue"]=completed.quantity*completed.unit_price
summary=completed.groupby("category",as_index=False).revenue.sum()
fig=px.bar(summary,x="category",y="revenue",title="Completed-order revenue by category (synthetic)")
fig.show()
# Write HTML for GitHub-friendly portfolio artifact.
Path("outputs").mkdir(exist_ok=True)
fig.write_html("outputs/retail_dashboard.html",include_plotlyjs=True)
print("Saved outputs/retail_dashboard.html")

## Business interpretation and limitations

In [ ]:
print("Customers:",len(customers),"Churn rate:",round(y.mean(),3))
print("Completed revenue:",round(completed.revenue.sum(),2))
print("Limitations: synthetic data; no prospective test; no causal inference; requires monitoring.")

## Reflection

1. What was your baseline? 2. What could go wrong in deployment? 3. Which results require human review?

Complete `Practice_Questions.md` before reading `Solutions.md`.